# XGBoost: four Ddc/Yb input ablations

Uses the same normalized datasets as the U-Net and transformer packages. Select case 30, 39, or 57. One scalar booster per target entry, with recoverable UBJ checkpoints.


In [1]:
from pathlib import Path
import subprocess
import sys
import json

CODE_DIR = Path.cwd()  # This extracted code folder
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 30
TARGET = 'dq'  # 'dq' or 'vmag'
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
DEVICE = 'cuda'  # or 'cuda:0'; use 'cpu' for local checks
ROUNDS = 600
MAX_DEPTH = 6
LEARNING_RATE = 0.05
EARLY_STOPPING = 20
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 1.0
THREADS = 4
SMOKE = False   # True tests tiny subsets and only 12 output entries
RESUME = False

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'gpu_xgboost_vmag.py').is_file(), 'Set CODE_DIR to the updated code folder'
for split in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{split}.npz').is_file()
outputs = CASE**2 if TARGET == 'dq' else CASE
print('Full run: up to', outputs * len(VARIANTS) * len(SEEDS), 'scalar models (constant outputs are skipped).')
print('Data:', DATA_DIR, '\nResults:', OUT_DIR)


Full run: up to 10800 scalar models (constant outputs are skipped).
Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30 
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case30_dq


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'gpu_xgboost_vmag.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--variants', *VARIANTS, '--target', TARGET, '--case', str(CASE),
    '--seeds', *map(str, SEEDS), '--device', DEVICE, '--threads', str(THREADS),
    '--rounds', str(ROUNDS), '--max-depth', str(MAX_DEPTH),
    '--learning-rate', str(LEARNING_RATE), '--early-stopping', str(EARLY_STOPPING),
    '--subsample', str(SUBSAMPLE), '--colsample-bytree', str(COLSAMPLE_BYTREE)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/gpu_xgboost_vmag.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case30_dq --variants ddc ddc_yb ddc_imag ddc_real --target dq --case 30 --seeds 42 123 2026 --device cuda --threads 4 --rounds 600 --max-depth 6 --learning-rate 0.05 --early-stopping 20 --subsample 0.8 --colsample-bytree 1.0
XGBoost 3.2.0 device cuda case 30 target dq

ddc seed=42: 900 features, 900 outputs; 0 completed
  completed 10/900 outputs
  completed 20/900 outputs
  completed 30/900 outputs
  completed 40/900 outputs
  completed 50/900 outputs
  completed 60/900 outputs
  completed 70/900 outputs
  completed 80/900 outputs
  completed 90/900 outputs
  completed 100/900 outputs
  completed 110/900 outputs
  completed 120/900 outputs
  completed 130/900 outputs
  complete

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/gpu_xgboost_vmag.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/XGBoost/results/case30_dq', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--case', '30', '--seeds', '42', '123', '2026', '--device', 'cuda', '--threads', '4', '--rounds', '600', '--max-depth', '6', '--learning-rate', '0.05', '--early-stopping', '20', '--subsample', '0.8', '--colsample-bytree', '1.0'], returncode=0)

In [3]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Full target benchmark:', summary['protocol']['complete_target'])
for variant, splits in summary['aggregate'].items():
    print(variant, 'test:', splits['test'])
print('Raw and standardized metrics:', OUT_DIR / 'comparison.csv')


Full target benchmark: True
ddc test: {'r2': {'mean': 0.8596210252330794, 'std_across_seeds': 4.262893042176363e-05}, 'rmse': {'mean': 1.614691349122171, 'std_across_seeds': 0.000245156839917965}, 'mae': {'mean': 0.6803706482175835, 'std_across_seeds': 0.00031293107932707064}, 'rmse_standardized': {'mean': 0.376631116653329, 'std_across_seeds': 5.718349633488444e-05}, 'mae_standardized': {'mean': 0.15869828557570032, 'std_across_seeds': 7.299199656657462e-05}}
ddc_yb test: {'r2': {'mean': 0.8610630651676136, 'std_across_seeds': 7.841007561341524e-05}, 'rmse': {'mean': 1.6063764694702147, 'std_across_seeds': 0.0004533073759863807}, 'mae': {'mean': 0.6769037202380527, 'std_across_seeds': 0.0003470388049594724}, 'rmse_standardized': {'mean': 0.37469164853302805, 'std_across_seeds': 0.00010573516445358012}, 'mae_standardized': {'mean': 0.1578896154087608, 'std_across_seeds': 8.094773669798327e-05}}
ddc_imag test: {'r2': {'mean': 0.8605828345962042, 'std_across_seeds': 3.5309286565782575e-0

## Interpretation

Raw-unit R2/RMSE/MAE are comparable with the updated neural-network packages on the same case/split/target. Standardized metrics are separately labelled. Smoke mode evaluates a subset of output entries and is not a full-matrix accuracy result. Test data does not choose the model rounds or seed. See README.md for inference and resume details.
